In [36]:
import pandas as pd
import numpy as np
import sys

sys.path.append("../src")

In [37]:
df = pd.read_csv("../data/balanced_reviews.csv")

print("Dataset Shape:")
print(df.shape)

df.head()

Dataset Shape:
(4000000, 6)


,product_id,product_title,category,review_text,rating,sentiment
0,4589130,Stainless Steel Blender,Home & Kitchen,Fast shipping and great packaging.,5,Positive
1,4716121,Long-Wear Matte Lipstick,Beauty,Highly recommend. Excellent quality.,5,Positive
2,9640962,Electric Toothbrush,Health & Personal Care,"Terrible experience, do not buy.",1,Negative
3,4442583,Hydrating Facial Serum,Beauty,"Does the job, but not impressed.",4,Positive
4,9757659,LEGO Building Kit,Toys & Games,Exactly what I needed!,5,Positive


In [38]:
print("\nColumns:")
print(df.columns.tolist())

print("\nSentiment Distribution:")
print(df["sentiment"].value_counts())

print("\nCategories:")
print(df["category"].value_counts())


Columns:
['product_id', 'product_title', 'category', 'review_text', 'rating', 'sentiment']

Sentiment Distribution:
sentiment
Positive    2063406
Neutral     1231769
Negative     704825
Name: count, dtype: int64

Categories:
category
Books                     501479
Health & Personal Care    500658
Beauty                    500256
Electronics               500089
Sports & Outdoors         499729
Toys & Games              499674
Fashion                   499280
Home & Kitchen            498835
Name: count, dtype: int64


In [39]:
# The balanced dataset has 26,400 reviews, which is optimal for training.
# We will sample safely, using up to 100,000 reviews.
SAMPLE_SIZE = min(100000, len(df))
df = df.sample(
    n=SAMPLE_SIZE,
    random_state=42
)

print(df.shape)

(100000, 6)


In [40]:
from text_normalizer import clean_text

df["clean_review"] = df["review_text"].apply(clean_text)

df[["review_text", "clean_review"]].head()

,review_text,clean_review
1049554,"Does the job, but not impressed.",does the job but not impressed
214510,Poor quality and broke quickly.,poor quality and broke quickly
2145764,Highly recommend. Excellent quality.,highly recommend excellent quality
2198867,Worst purchase I've made.,worst purchase ive made
1184366,Would not recommend to anyone.,would not recommend to anyone


In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=15000,
    stop_words="english"
)

X = tfidf.fit_transform(
    df["clean_review"]
)

print(X.shape)

(100000, 57)


In [42]:
y = df["sentiment"]

print(y.value_counts())

sentiment
Positive    51832
Neutral     30522
Negative    17646
Name: count, dtype: int64


In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(80000, 57)
(20000, 57)


In [44]:
from sklearn.svm import LinearSVC

model = LinearSVC(
    max_iter=1000,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

print("Training Complete")

Training Complete


In [45]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", accuracy)

Accuracy: 0.8284


In [46]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

    Negative       0.89      0.83      0.86      3589
     Neutral       0.70      0.78      0.73      6106
    Positive       0.90      0.86      0.88     10305

    accuracy                           0.83     20000
   macro avg       0.83      0.82      0.82     20000
weighted avg       0.84      0.83      0.83     20000



In [47]:
import joblib

joblib.dump(
    model,
    "../models/sentiment_model.pkl"
)

joblib.dump(
    tfidf,
    "../models/tfidf_vectorizer.pkl"
)

print("Saved Successfully")

Saved Successfully


In [48]:
sample_reviews = [

    "Amazing product. Highly recommended.",

    "Worst purchase ever.",

    "The product is okay."
]

sample_reviews = [
    clean_text(x)
    for x in sample_reviews
]

sample_vector = tfidf.transform(
    sample_reviews
)

predictions = model.predict(
    sample_vector
)

for review, pred in zip(
    sample_reviews,
    predictions
):
    print(review)
    print(pred)
    print("-"*50)

amazing product highly recommended
Positive
--------------------------------------------------
worst purchase ever
Negative
--------------------------------------------------
the product is okay
Positive
--------------------------------------------------


In [49]:
df["sentiment"].value_counts()

sentiment
Positive    51832
Neutral     30522
Negative    17646
Name: count, dtype: int64

In [50]:
sample_df = df.sample(
    100,
    random_state=42
)

sample_df.to_csv(
    "../data/sample_reviews.csv",
    index=False
)

print("sample_reviews.csv created")

sample_reviews.csv created


In [56]:
import pandas as pd

from dashboard_generator import generate_dashboard

df = pd.read_csv(
    "../data/analyzed_reviews.csv"
)

dashboard = generate_dashboard(df)

print(dashboard)

{'total_reviews': 100, 'positive_reviews': 53, 'neutral_reviews': 30, 'negative_reviews': 17, 'sentiment_summary': {'Positive': 53, 'Neutral': 30, 'Negative': 17}, 'issue_summary': {'Disappointment': 2, 'Packaging': 4, 'Quality': 5, 'Features': 4}, 'positive_features_summary': {'Performance': 19, 'Value for Money': 6, 'Quality': 22, 'Features': 6, 'Packaging': 7, 'Delivery': 7, 'Ease of Use': 3, 'Durability': 3}, 'category_summary': {'Sports & Outdoors': 17, 'Fashion': 15, 'Health & Personal Care': 14, 'Home & Kitchen': 13, 'Books': 12, 'Beauty': 11, 'Electronics': 11, 'Toys & Games': 7}, 'top_products': {'Noise-Canceling Headphones': 5.0, 'Board Game Bundle': 4.75, 'Multivitamin Pack': 4.5, 'Remote Control Car': 4.5, 'Cookbook with Recipes': 4.5, 'Natural Shampoo Set': 4.4, 'Memory Foam Pillow': 4.33, 'Resistance Exercise Bands': 4.2, 'Hydrating Facial Serum': 4.0, 'Slim Fit Denim Jeans': 4.0}, 'recent_negative_reviews': [{'product': 'Long-Wear Matte Lipstick', 'review': 'Very disappo

sentiment
Positive    2063406
Neutral     1231769
Negative     704825
Name: count, dtype: int64


Original Shape:
(4000000, 6)

Sentiment Distribution:
sentiment
Positive    2063406
Neutral     1231769
Negative     704825
Name: count, dtype: int64

Balanced Dataset Shape:
(600000, 6)

Balanced Distribution:
sentiment
Positive    200000
Neutral     200000
Negative    200000
Name: count, dtype: int64

Saved Successfully!


In [62]:
import pandas as pd

df = pd.read_csv("../data/raw_reviews.csv")

print("Rows:", len(df))
print("Unique Reviews:",
      df["Summary"].nunique())

print(
    df["Summary"]
    .value_counts()
    .head(20)
)

Rows: 205052
Unique Reviews: 92923
Summary
good                 17430
nice                 10571
good product          7191
nice product          5401
super                 3411
very good             2855
very nice             2760
very good product     1893
good quality          1825
ok                    1795
excellent             1643
awesome               1601
very nice product     1260
superb                 880
value for money        805
not good               799
best                   777
good one               722
excellent product      655
not bad                647
Name: count, dtype: int64
